<a href="https://colab.research.google.com/github/thatchangemaker1/agricycle-core/blob/main/agricycle_sim_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# AGRICYCLE USSD + SMS INTEGRATED SIMULATION
# ==========================================

import time

# --- 1. CONFIGURATION & SCIENTIFIC CONSTANTS ---
CARBON_FACTORS = {
    "1": 0.65,  # Maize Stalks & Cobs
    "2": 0.50,  # Maize Cobs Only
    "3": 0.40   # Groundnut Shells / Other Residues
}

# Simulated database of registered farms (storing profiles and phone numbers)
registered_farms = {
    "AGR-101": {"name": "Bwalya Chanda", "location": "Chongwe", "phone": "+260970000001"},
    "AGR-102": {"name": "Grace Mutale", "location": "Mazabuka", "phone": "+260960000002"}
}

# Simulated ledger for tracking all biomass and carbon transactions
biomass_ledger = []


# --- 2. SMS GATEWAY SIMULATION FUNCTION ---
def send_sms(phone_number, message_body):
    """
    Simulates sending an outbound SMS text message to the farmer's mobile phone.
    In production, this would interface with an API gateway like Africa's Talking or Twilio.
    """
    print("\n--------------------------------------------------")
    print(f" [SMS GATEWAY SIMULATION] Outbound to: {phone_number}")
    print(f" MESSAGE: {message_body}")
    print("--------------------------------------------------")
    time.sleep(1) # Simulate real-world network transmission delay


# --- 3. REGISTRATION & FALLBACK LOGIC ---

def register_new_farm():
    print("\n--- New Farm Quick Registration ---")
    new_name = input("Enter Full Name: ").strip()
    new_location = input("Enter District/Location: ").strip()
    phone_number = input("Enter Mobile Number (e.g., +260...): ").strip()

    # Generate a dynamic ID
    new_id = f"AGR-10{len(registered_farms) + 1}"

    registered_farms[new_id] = {
        "name": new_name,
        "location": new_location,
        "phone": phone_number
    }

    print(f"\n[USSD] Success! Farm registered for {new_name}.")

    # TRIGGER SMS CONFIRMATION
    sms_text = (
        f"Hello {new_name}, welcome to AgriCycle! "
        f"Your registration is confirmed. Your Farm ID is {new_id}. "
        f"Keep this ID safe to log your agricultural biomass and track carbon credits."
    )
    send_sms(phone_number, sms_text)

    run_ussd_session(new_id)


def verify_farm_id():
    print("\n==============================")
    print("      DIALING *384#...        ")
    print("==============================")
    farm_id = input("Enter your AgriCycle Farm ID (e.g., AGR-101): ").strip()

    if farm_id in registered_farms:
        farmer = registered_farms[farm_id]
        print(f"\n[USSD] Welcome back, {farmer['name']} ({farmer['location']})!")
        run_ussd_session(farm_id)
    else:
        print("\n[USSD] Error: Farm ID not recognized.")
        print("1. Register new farm profile")
        print("2. Re-enter Farm ID")
        choice = input("Select an option (1-2): ").strip()

        if choice == "1":
            register_new_farm()
        elif choice == "2":
            verify_farm_id()
        else:
            print("\n[USSD] Invalid selection. Session ended.")


# --- 4. PRODUCE SELECTION, LOGGING & RECEIPT SMS ---

def run_ussd_session(farm_id):
    farmer = registered_farms[farm_id]

    while True:
        print("\n--- AgriCycle USSD Main Menu ---")
        print("1. Log Organic Waste / Biomass")
        print("2. Check Carbon Credit Balance")
        print("3. Exit Session")

        choice = input("Choose an option (1-3): ").strip()

        if choice == "1":
            print("\n--- Select Produce / Residue Type ---")
            print("1. Maize Stalks & Cobs")
            print("2. Maize Cobs Only")
            print("3. Other Agricultural Residues (e.g., Groundnut Shells)")

            produce_choice = input("Select residue type (1-3): ").strip()

            if produce_choice not in CARBON_FACTORS:
                print("[USSD] Invalid produce selection. Returning to main menu.")
                continue

            try:
                weight_kg = float(input("Enter weight of biomass diverted (in kg): "))
                if weight_kg <= 0:
                    print("Weight must be greater than zero.")
                    continue

                # Apply the specific multiplier based on the produce chosen
                factor = CARBON_FACTORS[produce_choice]
                co2_offset = weight_kg * factor

                # Determine produce name for the transaction log and SMS
                produce_names = {
                    "1": "Maize Stalks & Cobs",
                    "2": "Maize Cobs Only",
                    "3": "Other Agricultural Residues"
                }
                selected_produce_name = produce_names[produce_choice]

                # Record transaction
                transaction = {
                    "farm_id": farm_id,
                    "produce": selected_produce_name,
                    "weight": weight_kg,
                    "co2_offset": co2_offset
                }
                biomass_ledger.append(transaction)

                print(f"\n[USSD] Success! Logged {weight_kg}kg of {selected_produce_name}.")
                print(f"[USSD] Estimated Carbon Offset: {co2_offset:.2f} kg CO2e.")

                # TRIGGER TRANSACTION RECEIPT SMS
                receipt_text = (
                    f"AgriCycle Update: Logged {weight_kg}kg of {selected_produce_name}. "
                    f"Estimated Carbon Credit Earned: {co2_offset:.2f} kg CO2e. "
                    f"Thank you for building a circular economy!"
                )
                send_sms(farmer["phone"], receipt_text)

            except ValueError:
                print("[USSD] Invalid input. Please enter a valid numerical value.")

        elif choice == "2":
            # Filter ledger entries for this specific farm
            farm_transactions = [t for t in biomass_ledger if t["farm_id"] == farm_id]
            farm_total_weight = sum(t["weight"] for t in farm_transactions)
            farm_total_co2 = sum(t["co2_offset"] for t in farm_transactions)

            print(f"\n--- Account Summary for {farm_id} ---")
            print(f"Total Biomass Diverted: {farm_total_weight} kg")
            print(f"Total Verified Carbon Offset: {farm_total_co2:.2f} kg CO2e")
            print(f"Total Transactions Logged: {len(farm_transactions)}")
            print("Status: Pending institutional audit & verification.")

        elif choice == "3":
            print(f"\n[USSD] Thank you, {farmer['name']}. Session closed.")
            break
        else:
            print("[USSD] Invalid choice. Please try again.")


# --- PROGRAM ENTRY POINT ---
if __name__ == "__main__":
    verify_farm_id()


      DIALING *384#...        


KeyboardInterrupt: Interrupted by user